# Fake News Detection

Classify news articles as **fake** or **real** using TF-IDF features and four scikit-learn classifiers
(Logistic Regression, Decision Tree, Gradient Boosting, Random Forest).

**Data:** download `Fake.csv` and `True.csv` from the
[Fake and Real News dataset](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset)
into the `data/` folder.

The reusable code (text cleaning, models, training, prediction) lives in the `fake_news` package, so this
notebook and the command-line scripts (`python -m fake_news.train` / `python -m fake_news.predict`) behave the same.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split

from fake_news import clean_text
from fake_news.model import FAKE, LABELS, MODEL_NAMES, REAL, build_models, predict

RANDOM_STATE = 0

## Load the data

In [ ]:
data_fake = pd.read_csv('data/Fake.csv')
data_true = pd.read_csv('data/True.csv')
data_fake['class'] = FAKE
data_true['class'] = REAL
data_fake.shape, data_true.shape

In [ ]:
data_fake.head()

In [ ]:
data_true.head()

Hold back the last 10 articles of each file for a manual sanity check at the end.
`.copy()` avoids pandas' `SettingWithCopyWarning`, and slicing with `iloc` works whatever the file length.

In [ ]:
N_MANUAL = 10
data_manual_testing = pd.concat([data_fake.iloc[-N_MANUAL:], data_true.iloc[-N_MANUAL:]]).copy()
data_fake = data_fake.iloc[:-N_MANUAL]
data_true = data_true.iloc[:-N_MANUAL]
data_fake.shape, data_true.shape

## Merge, de-duplicate and shuffle

In [ ]:
data = pd.concat([data_fake, data_true], ignore_index=True)[['text', 'class']]
print('Missing values:\n', data.isnull().sum(), sep='')

before = len(data)
data = data.dropna(subset=['text'])
data = data[data['text'].str.strip() != '']
data = data.drop_duplicates(subset=['text'])
print(f'Removed {before - len(data)} empty or duplicate articles')

data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
data.head()

In [ ]:
ax = sns.countplot(x=data['class'].map(LABELS))
ax.set(xlabel='', ylabel='Articles', title='Class balance')
plt.show()

## Clean the text

Almost every real article starts with a dateline such as `WASHINGTON (Reuters) - `, while fake ones don't.
Left in, the models learn "mentions Reuters ⇒ real" rather than anything about the content, which inflates
accuracy. `clean_text` strips that dateline, then lowercases and removes URLs, HTML tags, bracketed text,
tokens containing digits and punctuation.

In [ ]:
sample = data.loc[data['class'] == REAL, 'text'].iloc[0]
print('BEFORE:', sample[:300])
print()
print('AFTER: ', clean_text(sample)[:300])

## Train and evaluate

Each model is a `Pipeline` of `TfidfVectorizer(preprocessor=clean_text)` → classifier, so cleaning is applied
identically at training and prediction time. Gradient Boosting takes several minutes on the full dataset.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    data['text'], data['class'], test_size=0.25, random_state=RANDOM_STATE, stratify=data['class'])

models = build_models(random_state=RANDOM_STATE)
scores = {}
for name, model in models.items():
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    scores[name] = accuracy_score(y_test, pred)
    print(f'=== {MODEL_NAMES[name]} — accuracy {scores[name]:.4f} ===')
    print(classification_report(y_test, pred, target_names=[LABELS[FAKE], LABELS[REAL]]))

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 3.5))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, x_test, y_test, display_labels=['Fake', 'Real'], ax=ax, colorbar=False)
    ax.set_title(f'{MODEL_NAMES[name]}\n{scores[name]:.3f}')
plt.tight_layout()
plt.show()

## Manual testing on the held-out articles

In [ ]:
def manual_testing(news):
    results = predict(models, news)
    vote = results.pop('vote')
    for name, label in results.items():
        print(f'{MODEL_NAMES[name]:<20} {LABELS[label]}')
    print(f'Majority vote: {LABELS[vote]}')
    return vote

votes = data_manual_testing['text'].apply(lambda text: predict(models, text)['vote'])
print(f'Majority vote correct on {(votes == data_manual_testing["class"]).sum()} / {len(votes)} held-out articles')

Paste any article to classify it:

In [ ]:
news = input('Article text: ')
manual_testing(news)